In [2]:
import os, ray, sys
sys.path.append(".")

# === Ray/Modin Environment Variables ===
from init_ray_cluster import init_ray_cluster
# init_ray_clust


# === Core Libraries ===
# import modin.pandas as mpd
import pandas as pd
import matplotlib.pyplot as plt
import sys, glob, csv
import numpy as np
import networkx as nx
from datetime import timedelta, time
from time import gmtime, localtime, strftime, sleep
from lifelines import CoxPHFitter

# === SQL and Feather I/O ===
# import psycopg2
# from psycopg2 import sql, extras
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta
from collections import Counter
from IPython.display import IFrame, display, HTML

from functionsG import *

# from functionsG import (
#     get_params, run_sql_script, store_feather, load_feather,
#     categorize, reduce_floats, add_fy_col, print_syntax,
#     time_start, time_stop, print_syntax, run_pp
# )

# === Secure PostgreSQL Connection String and Create Engine ===
pp = run_pp()
from urllib.parse import quote_plus
safe_pw = quote_plus(pp)
params_dict = get_params()
conn_str =


# === Directories ===
load_dir = './chapter_folder'
load_local = './zz_load_local'
store_dir = './chapter_folder'
store_local = './zz_store_local'
var_dir = './running_vars'

# === Control Flags ===
rank = 'CPT'
optimize = False
build_df_work = True
keep_crl = True

# === Other Stuff ===
df_store_name = 'df_u'
df_save_dir = '/projects/TALENT_NET/big_dfs'
sql_path = './winbucket_link/'
data_table_pde = "study_talent_net.mv_master_ad_army_qtr_v3a"
data_table = "study_talent_net.mv_fouo_uz_master_ad_army_v3a"
good_years = None
null_entry = "NA"

SyntaxError: invalid syntax (1251103849.py, line 41)


## Boolean Control Flags

In [2]:

# =============================================
# Boolean Switches
# =============================================
SNAPS     = False ## PART SNAPS: Regenerate or load snapshot dates
ONEA      = False ## PART ONE A: Create snapshots Table
ONEB      = False ## PART ONE B: Create snapshots List
ONEC      = False ## PART ONE C: Create Snap Window Table
ONED      = False ## PART ONE D: Create PID Table
ONEE      = False ## PART ONE E: Build Base Table
ONEF      = False ## PART ONE F: Create DataFrame 'df_501_1st_base' from base table
ONEG      = False ## PART ONE G: Create Date of Birth Dictionary (dob_dict) and DataFrame (df_dob)
ONEH      = False ## PART ONE H: Create Exact Age DataFrame (df_exact_age) using (df_dob) DataFrame 
ONEI      = True ## PART ONE I: Add age_exact column to df_501_base using (df_exact_age) DataFrame 

THREE      = True ## PART THREE : Create & Save df_work
THREEA     = True ## PART THREE A: Eliminate pid_pde's with break-in-service while 'rank'
THREEB     = True ## PART THREE B: Eliminate pid_pde's with no date of rank for 'rank'
THREEC     = True ## PART THREE C: Eliminate pid_pde's with non-RA time while 'rank'.
THREED     = True ## PART THREE D: Generate work_pids_list and Table work_pids_{rank}

FOUR       = True ## PART FOUR : Assign Year Groups
FOURA      = True ## PART FOUR A: ID pids that have duplicate appointment dates after dropna
FOURB      = True ## PART FOUR B: Drop Duplicate apnt_dt pids (True) or use Mode (False)

FIVE      = True ## PART FIVE:  Assign Date of Rank for a particular rank
FIVEA      = True ## PART FIVE A: ID pids that have duplicate DOR for 'rank' after dropna
FIVEB      = True ## PART FIVE B: ID pids that have duplicate DOR for 'rank' after dropna

FIVEC      = True ## PART FIVE C: Find officers with Appointment dates AFTER CPT Dor
FIVED      = True ## PART FIVE D: Remove those pids from df_501_base, df_yg_dict, and df_gf
FIVEE      = True ## PART FIVE E: Save df_gf and variables from FOUR and FIVE

SIXA      = True ## PART SIX A: Build df_501_working
SIXB      = True ## PART SIX B: Map the yg, dor_cpt, age_cpt columns to df_501_working
SIXC      = True ## PART SIX C: Build df_work From df_501_working with pids from df_gf
SIXD      = True ## PART SIX D: Make df_work into df_501_filtered and push into the schema as Table '501_filtered'

SEVENA      = True ## PART SEVEN A: Visualize CPTs Who Became MAJ by YG (Count)
SEVENB      = True ## PART SEVEN B: Visualize CPTs Who Became MAJ by YG (Percent)

EIGHT    = True  ## PART EIGHT: Add Major Promotion Date and Major Age Columns


In [3]:
col_list_base = ['snpsht_dt','pid_pde','compo', 'ofcr_apnt_dt',
                 
                 'rank_pde','ppln_pgrd_eff_dt',
                                
                 'edu_tier_cd','edu_lvl_cd','grad_pro_edu_stat_cd',
                 
                 'occ_crer_grp_cd','dty_svc_occ_cd','pri_dod_occ_cd','pri_svc_occ_cd','dty_dod_occ_cd',

                 'pn_sex_cd','race_cd','eth_aff_cd','pn_age_qy','faith_grp_cd', 'mrtl_stat_cd',

                 'asg_uic_pde','prm_dty_stn_dprt_dt','prm_dty_stn_arrv_dt','ofcr_act_stat_pe_dt', 'date_birth_pde']
hello_world()

columns = ["pid_pde", "time", "event", "pn_age_qy", "pn_sex_cd", "mrtl_stat_cd",
           "occ_crer_grp_cd", "edu_lvl_cd","faith_grp_cd","race_cd","ofcr_act_stat_pe_dt","date_birth_pde","asg_uic_pde"]

## Set Variabes and Parameters

In [4]:
# ---1. Define parameters ---
study_schema = 'study_talent_net.'
table_name = data_table_pde
default_schema = 'study_talent_net_shared.' 
column_list = col_list_base
pids_table_name = default_schema+'b_501_pids'
pids_table = default_schema+'pids_o_culld'
#### OPTION TO GET ALL PIDS
# pids_table = 'pids_o'
#### OPTION TO GET ALL PIDS
df_name = 'df_501_base'
sql_table_name = 'b_501_base'
window_tup = (2000,2022)
snap_window_table_name = f'b_501_snap_window_{window_tup[0]}_{window_tup[1]}'
where_clause = """
    WHERE 
        base.rank_grp_pde IN ('OJ','OS') 
        --AND base.compo IN ('R')
    """
join_clause_1 = f"""
        INNER JOIN {pids_table} AS pidso
            ON base.pid_pde = pidso.pid_pde"""
# join_clause_1 = " -- "
join_clause_2 = f"""
        INNER JOIN {default_schema+snap_window_table_name} AS win
            ON base.snpsht_dt::date = win.snpsht_dt::date"""
# join_clause_2 = " -- "
dict_501 = {'table_name':table_name,
           'column_list':column_list,
           'where_clause':where_clause,
           'join_clause_1':join_clause_1,
           'join_clause_2':join_clause_2,
           'df_name':df_name,
           'sql_table_name':sql_table_name,
           'window_tup':window_tup,
           'snap_window_table_name':snap_window_table_name,
           'pids_table_name': pids_table_name,
           'pids_table': pids_table}

# ---2. Create SQLAlchemy Engine ---
engine = create_engine(conn_str)
check_sqlalchemy_connection(engine)


 Valid SQLAlchemy Engine.


## Some Functions

In [5]:
def time_begin(section_name, nest=0):
    import time
    print("  " * nest + f"\u25B6 START: {section_name}")
    return time.time()

def time_end(start_time, nest=0):
    import time
    elapsed = time.time() - start_time
    print("  " * nest + f"\u23F9 END: Elapsed time: {hms_string(elapsed)} at {tymeout()}.")


In [6]:
def fast_copy(df, table_name, schema='AN_LEVINEC', nest=0):
    with engine.connect() as conn:
        df.head(0).to_sql(table_name,
                  con=conn,
                  if_exists='replace',
                  index=False,
                  method='multi')
        conn.commit()
    import psycopg2
    from io import StringIO
    actfscp = time_start(f"Pushing DataFrame to {schema}.{table_name}.")
    fscp = time_start(actfscp,nest = nest+1,show=True)
    buffer = StringIO()
    df.to_csv(buffer, index=False, header=False)
    buffer.seek(0)
    conn = psycopg2.connect(**get_params())
    with conn.cursor() as cursor:
        cursor.copy_expert(f"COPY {schema}.{table_name} FROM STDIN WITH CSV", buffer)
        conn.commit()
    conn.close()
    time_stop(fscp,action=actfscp,nest=nest+1)

def list_compare(list1, list2, show=True):
    set1 = set(list1); set2 = set(list2)
    if show:
        print(f"There are {len(set1):,} unique items in list 1.")
        print(f"There are {len(set2):,} unique items in list 2.")
    one_not_two = [item for item in set1 if item not in set2]
    if show:
        print(f"There are {len(one_not_two):,} unique items in list 1 not in list 2.")
    two_not_one = [item for item in set2 if item not in set1]
    if show:
        print(f"There are {len(two_not_one):,} unique items in list 2 not in list 1.")
    one_and_two = [item for item in set1 if item in set2]
    if show:
        print(f"There are {len(one_and_two):,} unique items in list 1 intersect list 2.")
    return one_not_two, two_not_one, one_and_two

def get_years_diff(begin_dt,end_dt):
    time_diff = abs(end_dt - begin_dt)
    total_days = time_diff.days
    years_diff = total_days / 365.2425
    return round(years_diff,2)

def ray_cleanup(object_refs: list = None, verbose: bool = True):
    """
    Frees Ray object references and runs garbage collection.
    Does NOT shut down or re-sart Ray.
    Safe to use between pipeline steps if Ray session should
    rmeain alive.
    """
    import ray, gc
    if object_refs:
        for ref in object_refs:
            try:
                ray.internal.free(ref)
                if verbose:
                    print("Freed Ray object: {ref}")
            except Exception as e:
                if verbose:
                    print(f"Ray cleanup failed for object: {e}")
    if verbose:
        print("Running garbage collection...")
        gc.collect()
    if verbose:
        print("Memory cleanup complete.")  

In [7]:
def check_table(table_name, show=True):
    with engine.connect() as conn:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}"))
        row_count = result.scalar()
        if show:
            print(f"Table {table_name} is accessible and has {row_count:,} rows.")
    return row_count

In [8]:
def table_into_df(sql_table_name,nest,schema=default_schema,modin=True,show=True):
    # === Load the table into DataFrame ===
    row_count = check_table(schema+sql_table_name,show=show)
    action_p = (f"Reading the table '{schema+sql_table_name}' with {row_count:,} rows into pandas DataFrame.")
    pandy = time_start(action_p,nest=nest+1,show=show)
    try:
        if modin:
            init_ray_cluster()
            import modin.pandas as mpd
            engine = create_engine(conn_str)
            df = mpd.read_sql(F"SELECT * FROM {schema+sql_table_name}",engine)
            import pandas as pd
            df_out = df._to_pandas()
            del df
            ray_cleanup()
            ray.shutdown()
            
        else:
            import pandas as pd
            engine = create_engine(conn_str)
            df_out = pd.read_sql(F"SELECT * FROM {schema+sql_table_name}",engine)
        time_stop(pandy,action = f"To read {row_count:,} rows..",nest=nest+1,show=show)
        return df_out

    except Exception as e:
        time_stop(pandy,action = f"To read {row_count:,} rows..")
        print(f"Failed to read {schema+sql_table_name}: {e}")

In [9]:
def load_lu_tables(nest,lu_table_list=['lkup_army_mil_occ_grp',
                                       'lkup_army_mos',
                                       'lkup_atrrs_output_status_cd',
                                       'lkup_dmdc_civ_edu_lvl_cd',
                                       'lkup_dmdc_mil_edu_lvl_cd',
                                       'lkup_dmdc_deers_mrtl_stat_cd'],schema='pde_lookup.',show=True):
    lu_table_dict = dict()
    import pandas as pd
    for tab in lu_table_list:
        with engine.connect() as conn:
            if show:
                print(F"Loading '{schema+tab}' into 'df_{tab}'.")
            lu_table_dict['df_'+tab] = table_into_df(tab,nest+1,schema=schema,modin=False,show=show)
    print(f"'lu_table_dict' loaded with the following DataFrames: \n{list(lu_table_dict.keys())}")
    return lu_table_dict


lu_table_dict = load_lu_tables(0,show=False)
df_lkup_army_mos = lu_table_dict['df_lkup_army_mos']
df_lkup_army_mil_occ_grp = lu_table_dict['df_lkup_army_mil_occ_grp']
df_lkup_dmdc_deers_mrtl_stat_cd = lu_table_dict['df_lkup_dmdc_deers_mrtl_stat_cd']

'lu_table_dict' loaded with the following DataFrames: 
['df_lkup_army_mil_occ_grp', 'df_lkup_army_mos', 'df_lkup_atrrs_output_status_cd', 'df_lkup_dmdc_civ_edu_lvl_cd', 'df_lkup_dmdc_mil_edu_lvl_cd', 'df_lkup_dmdc_deers_mrtl_stat_cd']


In [10]:
def time_begin(section_name, nest=0):
    import time
    print("  " * nest + f"\u25B6 START: {section_name}")
    return time.time()

def time_end(start_time, nest=0):
    import time
    elapsed = time.time() - start_time
    print("  " * nest + f"\u23F9 END: Elapsed time: {hms_string(elapsed)} at {tymeout()}.")


def get_mode_dict_for_col(df_in, col, nest, rank=None):
    t0 = time_begin(f"Get mode_dict for '{col}'.", 0)
    df_w = df_in.copy()
    if rank:
        df_w = df_w[df_w.rank_pde == rank]
    elif col == 'ppln_pgrd_eff_dt':
        print(f"!! A Rank argument is required for a {col} mode!!")
        time_end(t0, 0)
        return
    df_w = df_w.dropna(subset=[col])
    print(f"Getting mode_dict...")
    print(f" after {col} dropna df_w has {df_w.pid_pde.nunique():,} pids")
    if col == 'ofcr_apnt_dt':
        new_col = 'yg'
        df_w[col] = df_w[col].apply(get_fy).astype(int)
        print(f" after apply(get_fy) df_w has {df_w.pid_pde.nunique():,} pids")
        print(f" after apply(get_fy) df_w has {df_w[col].nunique():,} {col}'s")
        print(f"Null test: ",[val for val in df_w[col].unique().tolist() if val !=val])
        # print(f"Set of values for df_w['{col}']:",set(df_w[col].unique()))
    print(f"Generatiog mode_result...")    
    mode_result = df_w.groupby('pid_pde')[col].agg(lambda x: x.mode().iloc[0])
    print(f" after mode_result, mode_result has {len(mode_result):,} pids")
    
    mode_dict = mode_result.to_dict()
    print(f" after mode_dict, mode_dict has {len(mode_dict):,} pids")
    
    print(f" after mode_dict, mode_dict has {df_w[col].nunique():,} unique values for '{col}'")
    print(f" and {len(mode_dict):,} unique 'pid_pde' values.")
    print(f"Null test: ",[val for val in mode_dict.values() if val !=val])
    # print(f"Set of values for df_w['{col}']:",set(df_w[col].unique()))
    # print(f"Set of values for mode_dict:",set(mode_dict.values()))
    time_end(t0, 0)
    return mode_dict

## PART ONE A: Create snapshots Table

In [11]:
## PART ONE A: Create snapshots Table
def build_snapshots_table(nest,table_name=table_name,show=True):
    snapshots_table_name = 'snapshots'
    sql = f"""\n\n\n
    
    DROP TABLE IF EXISTS {default_schema+snapshots_table_name};\n
    CREATE TABLE {default_schema+snapshots_table_name} AS 
    SELECT DISTINCT snpsht_dt FROM {table_name}
    ORDER BY snpsht_dt;"""
    
    sql2 = f"""
    CREATE INDEX idx_snpsht_dt_on_{snapshots_table_name} ON {default_schema+snapshots_table_name}(snpsht_dt);"""
    
    # === Execute Create Table ===
    action_snp = (f"Creating the table {snapshots_table_name} in my schema from {table_name}.")
    schema_start = time_start(action_snp,nest=nest+1,show=show)
    
    try:
        print_syntax(sql)
        with engine.connect() as conn:
            conn.execute(text(sql))
            conn.commit()
        print("")
        print_syntax(sql2)
        with engine.connect() as conn:
            conn.execute(text(sql2))
            conn.commit()
            
    except Exception as e:
        print(f"SQL execution failed for {default_schema+snapshots_table_name}: {e}.")
        return f"Failed {default_schema+snapshots_table_name} (SQL)1"
        
    check_table(default_schema+snapshots_table_name)
    
    time_stop(schema_start,action=action_snp, nest=nest+1)

if SNAPS and ONEA:
    build_snapshots_table(0)

## PART ONE B: Create snapshots List

In [12]:
## PART ONE B: Create snapshots List
def build_snapshots_list(save=True):
    with engine.connect() as conn:
        df_snapshots = pd.read_sql_table(table_name='snapshots',con=conn,schema=default_schema.replace('.',''))
        conn.commit()
    snapshots = sorted(list(df_snapshots.snpsht_dt))
    print(f"Length of df_snapshots is {len(snapshots)}")
    store_pickle(snapshots,'snapshots', var_dir)
    return snapshots

if SNAPS and ONEB:
    snapshots = build_snapshots_list()
else:
    snapshots = load_pickle('snapshots',var_dir)

## PART ONE C: Create Snap Window Table

In [13]:
## PART ONE C: Create Snap Window Table
def build_snap_window_table(dict_501, nest, show=True):
    lo_yr, hi_yr = dict_501['window_tup']
    actionrt = f"Reading table 'snapshots' into DataFrame."
    rdtab = time_start(actionrt,nest=nest+1,show=show)
    with engine.connect() as conn:
        df_snapshots = pd.read_sql_table(table_name='snapshots',con=conn,schema=default_schema.replace('.',''))
        conn.commit()
    time_stop(rdtab,action=actionrt, nest=nest+1)
    df_snapshots = add_fy_col(df_snapshots,show=False)
    df_snap_window = df_snapshots[df_snapshots['fy'].between(lo_yr,hi_yr)].copy()
    df_snap_window.drop(columns=['fy'], inplace=True)
    snap_window_table_name = dict_501['snap_window_table_name']
    actionrdf = f"Writing DataFrame 'df_snap_window' to Table '{snap_window_table_name}' in my schema. ({default_schema+snap_window_table_name})"
    wrttab = time_start(actionrdf,nest=nest+1,show=show)
    with engine.connect() as conn:
        df_snap_window.to_sql(snap_window_table_name,
                              con=conn,
                              schema=default_schema.replace('.',''),
                              if_exists='replace',
                              index=False,
                              method='multi')
        conn.commit()
    
    
    # Add index on snpsht_dt
    actionswtidx = "Indexing Table"
    start_idx = time_start(actionswtidx,nest=nest+2)
    sql_idx = f'CREATE INDEX idx_snpsht_dt_on_{snap_window_table_name} ON {default_schema+snap_window_table_name}(snpsht_dt);'
    print_syntax(sql_idx,lang='sql')
    with engine.connect() as conn:
        conn.execute(text(sql_idx))
        conn.commit()
    
    time_stop(start_idx,action=actionswtidx, nest=nest+2)
    time_stop(wrttab,action=actionrdf, nest=nest+1)
    check_table(default_schema+snap_window_table_name)
    return df_snap_window

if ONEC:
    df_snap_window = build_snap_window_table(dict_501,0)

## PART ONE D: Create PID Table

In [14]:
## PART ONE D: Create PID Table
def create_pids_table(dict_501,nest,show=True):
    pids_table_name = dict_501['pids_table_name']
    pids_table_name_clean = pids_table_name.split('.')[1]
    pids_table = dict_501['pids_table']
    sql = f"""\n\n\n

    DROP TABLE IF EXISTS {pids_table_name};\n
    CREATE TABLE {pids_table_name} AS 
    SELECT pid_pde FROM {pids_table}
    ORDER BY pid_pde;
    """
    
    sql2 = f"""
    CREATE INDEX idx_pid_pde_on_{pids_table_name_clean} ON {pids_table_name}(pid_pde);"""
    print_syntax(sql2)
    # === Execute Create Table ===
    action_snp = (f"Creating the table {pids_table_name_clean} in my schema from {pids_table}.")
    schema_start = time_start(action_snp,nest=nest+1,show=show)
    
    try:
        print_syntax(sql)
        with engine.connect() as conn:
            conn.execute(text(sql))
            conn.commit()
        print("")
        print_syntax(sql2)
        with engine.connect() as conn:
            conn.execute(text(sql2))
            conn.commit()
            
    except Exception as e:
        print(f"SQL execution failed for {pids_table_name}: {e}.")
        return f"Failed {pids_table_name} (SQL)1"
        
    check_table(pids_table_name)
    
    time_stop(schema_start,action=action_snp, nest=nest+1)    

if ONED:
    create_pids_table(dict_501,0)


## PART ONE E: Build Base Table

In [15]:
dict_501

{'table_name': 'study_talent_net.mv_master_ad_army_qtr_v3a',
 'column_list': ['snpsht_dt',
  'pid_pde',
  'compo',
  'ofcr_apnt_dt',
  'rank_pde',
  'ppln_pgrd_eff_dt',
  'edu_tier_cd',
  'edu_lvl_cd',
  'grad_pro_edu_stat_cd',
  'occ_crer_grp_cd',
  'dty_svc_occ_cd',
  'pri_dod_occ_cd',
  'pri_svc_occ_cd',
  'dty_dod_occ_cd',
  'pn_sex_cd',
  'race_cd',
  'eth_aff_cd',
  'pn_age_qy',
  'faith_grp_cd',
  'mrtl_stat_cd',
  'asg_uic_pde',
  'prm_dty_stn_dprt_dt',
  'prm_dty_stn_arrv_dt',
  'ofcr_act_stat_pe_dt',
  'date_birth_pde'],
 'where_clause': "\n    WHERE \n        base.rank_grp_pde IN ('OJ','OS') \n        --AND base.compo IN ('R')\n    ",
 'join_clause_1': '\n        INNER JOIN study_talent_net_shared.pids_o_culld AS pidso\n            ON base.pid_pde = pidso.pid_pde',
 'join_clause_2': '\n        INNER JOIN study_talent_net_shared.b_501_snap_window_2000_2022 AS win\n            ON base.snpsht_dt::date = win.snpsht_dt::date',
 'df_name': 'df_501_base',
 'sql_table_name': 'b_501_

In [16]:
## PART ONE E: Build Base Table
def build_base_table(dict_501, nest, show=True):
    # === Imports ===
    from sqlalchemy import create_engine, text
    from functionsG import time_start, time_stop, print_syntax, store_feather
    from functionsG import categorize, reduce_floats, add_fy_col
    # import modin.pandas as mpd
    import pandas as pd
    import gc

    
    # === Unpack Inputs ===
    sql_table_name = default_schema+dict_501['sql_table_name']
    table_name = dict_501['table_name']
    column_list = dict_501['column_list']
    join_clause_1 = dict_501['join_clause_1']
    join_clause_2 = dict_501['join_clause_2']
    where_clause = dict_501['where_clause']
    pids_table_name = dict_501['pids_table_name']
    
    engine = create_engine(conn_str)
    check_sqlalchemy_connection(engine)
    
       # === Build SQL to select only needed columns ===
    sql = f"""\n
    
    DROP TABLE IF EXISTS {sql_table_name};\n
    CREATE TABLE {sql_table_name} AS 
    SELECT {', '.join(['base.'+col for col in column_list])}\n 
    FROM {table_name} AS base
    {join_clause_1}
    {join_clause_2}
    {where_clause}       
    ORDER BY base.pid_pde, base.snpsht_dt;
    """
    
    sql2 = f"""\n
    CREATE INDEX idx_snpsht_dt_on_{dict_501['sql_table_name']} ON {sql_table_name}(snpsht_dt);
    CREATE INDEX idx_pid_pde_on_{dict_501['sql_table_name']} ON {sql_table_name}(pid_pde);
    CREATE INDEX idx_asg_uic_pde_on_{dict_501['sql_table_name']} ON {sql_table_name}(asg_uic_pde);"""
    
    # === Execute Create Table ===
    actionbld = f"Creation of '{sql_table_name}' using '{table_name}', '{pids_table_name}', and '{snap_window_table_name}'."
    bldbase = time_start(actionbld,nest=nest+1,show=show)
    try:
        actionbsj = f"Selection and joining  of '{sql_table_name}'."
        ssj = time_start(actionbsj,nest=nest+2,show=show)
        print_syntax(sql)
        with engine.connect() as conn:
            conn.execute(text(sql))
            conn.commit()
        print("")
        time_stop(ssj,action=actionbsj, nest=nest+2)
        actionbidx2 = f"Indexing of '{sql_table_name}'."
        ssidx = time_start(actionbidx2,nest=nest+2,show=show)
        print_syntax(sql2)
        with engine.connect() as conn:
            conn.execute(text(sql2))
            conn.commit()
        time_stop(ssidx,action=actionbidx2, nest=nest+2)
    except Exception as e:
        print(f"SQL execution failed for {sql_table_name}: {e}.")
        return f"Failed {df_name} (SQL)1"
    
    actionct = f"Table check on {sql_table_name}."
    chktbl = time_start(actionct,nest=nest+2,show=show)  
    check_table(sql_table_name)
    time_stop(chktbl,action=actionct,nest=nest+2)
    time_stop(bldbase,action=actionbld,nest=nest+1)

    return f"Success {sql_table_name}"   

if ONEE:
    build_base_table(dict_501, 0, show=True)

In [17]:
def cut_pids(df_in,cut_pid_list):
    df_out = df_in[~df_in['pid_pde'].isin(cut_pid_list)]
    return df_out

def find_break_in_service_pids(df_in,full_snapshot_list,rank,nest,show=True):
    break_in_service_pids = []; no_rank_dates = []
    # df_working = df_in.sort_values(by=['pid_pde','snpsht_dt'])
    df_working = df_in[df_in.rank_pde == rank]

    for pid, group in df_working.groupby('pid_pde',observed=True):
        rank_dates = sorted(group['snpsht_dt'].unique())
        if not rank_dates:
            no_rank_dates.append(pid)
        else:
            expected_dates = [d for d in full_snapshot_list if rank_dates[0] <= d <= rank_dates[-1]]
        if set(rank_dates) != set(expected_dates):
            break_in_service_pids.append(pid)
    bisp = sorted(list(set(break_in_service_pids)))
    nrd = sorted(list(set(no_rank_dates)))
    print(f"A total of {len(bisp):,} break in service pids and {len(nrd):,} no rank dates pids were found for {rank}'s.")
    return bisp, nrd

def get_fy(date_in,make_int=True):
    try:
        date_ts = pd.to_datetime(date_in)
        fy = date_ts.year + 1 if date_ts.month>= 10 else date_ts.year
        if make_int:
            fy = int(fy)
        return fy
    except Exception as e:
        return f"Error processing {date_in}: {e}"

## PART ONE F: Create DataFrame 'df_501_1st_base' from base table

In [18]:
if ONEF:
    df_501_1st_base = table_into_df(dict_501['sql_table_name'],0,modin=False)
    sleep(2)
    if optimize:
        display(df_501_1st_base.dtypes)
        df_501_1st_base = optimize_dtypes(df_501_1st_base)
        display(df_501_1st_base.dtypes)
    store_feather(df_501_1st_base,'df_501_1st_base')
else:
    df_501_1st_base = load_feather('df_501_1st_base')

 df_501_1st_base Loaded!!  - (01.58 seconds and 908.806 MB of memory)


## PART ONE G: Create Date of Birth Dictionary (dob_dict) and DataFrame (df_dob)

In [19]:
def create_dob_df_and_dict(df_in):
    df= df_in.copy()
    print(f"The input DataFrame has {df.pid_pde.nunique():,} unique pids:")
    dob_dict = get_mode_dict_for_col(df, 'date_birth_pde',0)
    print(f"Final date of birth dictionary (dob_dict) has {len(dob_dict):,} pid_pde keys.")
    print(f"Storing Final date of birth dictionary (dob_dict)...")
    store_pickle(dob_dict,'dob_dict',var_dir)
    df_dob = pd.DataFrame(dob_dict.items(), columns = ['pid_pde','dob'])
    print(f"Storing date of birth DataFrame ('df_dob')...")
    store_feather(df_dob,'df_dob')
    
if ONEG:
    create_dob_df_and_dict(df_501_1st_base)
else:
    dob_dict = load_pickle('dob_dict',var_dir)
    df_dob = load_feather('df_dob')

 df_dob Loaded!!  - (00.03 seconds and 2.310 MB of memory)


## PART ONE H: Create Exact Age DataFrame (df_exact_age) using (df_dob) DataFrame 

##### ref_col ----------> This is the column that will be running referenced
##### static_col -------> This column value should not change
##### pos_dup_val_col --> This should be a static column, but may have duplicate values
##### transition_col ---> This column is a temporary bridge
##### after_col --------> When moving columns, this is the DataFrame column after which you're moving the column

In [20]:
## PART ONE H: Create Exact Age DataFrame (df_exact_age) using (df_dob) DataFrame 
def create_df_exact_age(df_in_name='df_501_1st_base',
                        id_col='pid_pde',
                        new_col='age_exact',
                        ref_col='snpsht_dt',
                        static_col ='date_birth_pde',
                        df_dob_name='df_dob',
                        after_col='pn_age_qy'):
    df = load_feather(df_in_name); df_dob = load_feather(df_dob_name)
    transition_col=df_dob.columns.tolist()[1]
    print(f"Transition Column: {transition_col}.")

    print(f"Creating the 'df_exact_age' DataFrame using ('{df_in_name}') and the '{df_dob_name}' DataFrame columns...")
    print(f"\n\nThe input DataFrame, '{df_in_name}' has {df.pid_pde.nunique():,} unique pids:")
    
    print(f"Creating the transition DataFrame 'df_merge'")
    df_merge = df[df[id_col].isin(df_dob[id_col].tolist())][[ref_col,id_col]]
    # display(df_merge.head(4))
    
    print(f"Merging the transition DataFrame with '{df_dob_name}'")
    df_merge = df_merge.merge(df_dob, on=id_col)
    # display(df_merge.head(4))

    goto1 = time_begin(f"Computing the '{new_col}' column from the '{ref_col}' and '{transition_col}' columns..")
    df_merge[new_col] = df_merge.apply(lambda row: get_years_diff(row[transition_col], row[ref_col]), axis=1)
    time_end(goto1)

    df_exact_age = df_merge.drop(columns=[transition_col])
    df_exact_age.head(5)

    store_feather(df_exact_age,'df_exact_age')
    # df_exact_age = load_feather('df_exact_age')
    print(f"The Exact Age DataFrame, 'df_exact_age' has {df_exact_age.pid_pde.nunique():,} unique pids:")
    # display("df_exact_age:",df_exact_age.head(4))
    return df_exact_age

if ONEH:
    df_exact_age = create_df_exact_age()
else:
    df_exact_age = load_feather('df_exact_age')

 df_exact_age Loaded!!  - (00.16 seconds and 109.051 MB of memory)


## PART ONE I: Add age_exact column to df_501_base using (df_exact_age) DataFrame

In [21]:
## PART ONE I: Add age_exact column to df_501_base using (df_exact_age) DataFrame
def add_exact_age_column(df_in_name,after_col='pn_age_qy',ref_col='snpsht_dt'):
    id_col = 'pid_pde'
    df_age_e = df_exact_age.copy()
    new_col = df_age_e.columns[2]
    df = load_feather(df_in_name)
    display("df columns before merge:",df.columns)
    goto2 = time_begin(f"Merging the transition DataFrame ('df_exact_age') onto '{df_in_name}' using left join on the '{id_col}' and '{ref_col}' columns..")
    df = df.merge(df_age_e, how='left', on=[ref_col,id_col])
    df = move_column_after(df,new_col,after_col)
    display("df columns after merge:",df.columns)
    time_end(goto2)
    # display(df[[ref_col, id_col, after_col, new_col]])
    
    no_exact_age_pids = df[df[new_col].isna()][id_col].unique().tolist()
    print(f"The updated DataFrame ('{df_in_name}') with the added '{new_col}' column has {len(no_exact_age_pids):,} individuals with 'NaT' values for '{new_col}'.")
    return df
if ONEI:
    df_in_name = 'df_501_1st_base'
    df_501_base = add_exact_age_column(df_in_name)
    store_feather(df_501_base,'df_501_base')
else:
    df_501_base = load_feather('df_501_base')

 df_501_1st_base Loaded!!  - (01.47 seconds and 908.806 MB of memory)


'df columns before merge:'

Index(['snpsht_dt', 'pid_pde', 'compo', 'ofcr_apnt_dt', 'rank_pde',
       'ppln_pgrd_eff_dt', 'edu_tier_cd', 'edu_lvl_cd', 'grad_pro_edu_stat_cd',
       'occ_crer_grp_cd', 'dty_svc_occ_cd', 'pri_dod_occ_cd', 'pri_svc_occ_cd',
       'dty_dod_occ_cd', 'pn_sex_cd', 'race_cd', 'eth_aff_cd', 'pn_age_qy',
       'faith_grp_cd', 'mrtl_stat_cd', 'asg_uic_pde', 'prm_dty_stn_dprt_dt',
       'prm_dty_stn_arrv_dt', 'ofcr_act_stat_pe_dt', 'date_birth_pde'],
      dtype='object')

▶ START: Merging the transition DataFrame ('df_exact_age') onto 'df_501_1st_base' using left join on the 'pid_pde' and 'snpsht_dt' columns..


'df columns after merge:'

Index(['snpsht_dt', 'pid_pde', 'compo', 'ofcr_apnt_dt', 'rank_pde',
       'ppln_pgrd_eff_dt', 'edu_tier_cd', 'edu_lvl_cd', 'grad_pro_edu_stat_cd',
       'occ_crer_grp_cd', 'dty_svc_occ_cd', 'pri_dod_occ_cd', 'pri_svc_occ_cd',
       'dty_dod_occ_cd', 'pn_sex_cd', 'race_cd', 'eth_aff_cd', 'pn_age_qy',
       'age_exact', 'faith_grp_cd', 'mrtl_stat_cd', 'asg_uic_pde',
       'prm_dty_stn_dprt_dt', 'prm_dty_stn_arrv_dt', 'ofcr_act_stat_pe_dt',
       'date_birth_pde'],
      dtype='object')

⏹ END: Elapsed time: 03.97 seconds at 20:13:37 (EST) Sun, 30 Nov 2025.
The updated DataFrame ('df_501_1st_base') with the added 'age_exact' column has 29 individuals with 'NaT' values for 'age_exact'.
 df_501_base Stored!!  - (06.05 seconds and 945.159 MB of memory)


## PART THREE A, THREE B, THREE C : Create 'df_work' (only CPTs)

In [ ]:
## PART THREE A, THREE B, THREE C : Create 'df_work'

## Here we create 'df_work_{rank}' which will be a DataFrame with ONLY Captains
# Designed to be used for our 'who made major' machine.  
# We'll get rid of all pids with a break in service while the target rank (Captain) 
# as well as all pids that have some non-RA time while the target rank.

def create_df_work(df_in, rank, nest):
    # --- 1. Create 'df_work' by filtering just df_501_base rows with 'rank' for rank_pde
    print(f"---> 1. Creating 'df_wg_{rank}' by filtering just df_501_base rows with '{rank}' for rank_pde.")
    df_wg = df_in.copy()
    df_len = len(df_wg)
    df_pids_num = df_wg.pid_pde.nunique()
    print(f"'df_in' has {df_pids_num:,} unique pid_pde's and {df_len:,} rows.\n")
   
    df_wg = df_wg[['snpsht_dt', 'pid_pde', 'compo', 'ofcr_apnt_dt', 'rank_pde', 'ppln_pgrd_eff_dt',
                   'occ_crer_grp_cd', 'asg_uic_pde','ofcr_act_stat_pe_dt','mrtl_stat_cd',
                   'date_birth_pde']]
    df_wg = df_wg[df_wg.rank_pde == rank]
    if optimize:
        df_wg['rank_pde'].cat.remove_unused_categories()
    df_len = len(df_wg)
    df_pids_num = df_wg.pid_pde.nunique()
    print(f"'df_wg' has {df_pids_num:,} unique pid_pde's and {df_len:,} rows.\n")
    
    
    print(f"---> 2. Identifying pid_pde's that have a break in service or no date of rank while '{rank}'.")
    if THREEA or THREEB:
        break_in_service_pids, no_rank_dates = find_break_in_service_pids(df_wg,snapshots,rank,nest=nest+1,show=True)
    # --- 2. Eliminate pid_pde's that have a break in service while 'rank'.
        if THREEA and break_in_service_pids:
            pre_cut_pids_num = df_wg.pid_pde.nunique()
            df_wg = cut_pids(df_wg, break_in_service_pids)
            post_cut_pids_num = df_wg.pid_pde.nunique()
            print(f"   'df_wg' has {len(break_in_service_pids):,} with a break-in-service while {rank}.\n \
                They are being eliminated for a {redux_perc(pre_cut_pids_num,post_cut_pids_num)} reduction.")
            print(f"'df_wg' now has {post_cut_pids_num:,} unique pid_pde's.\n")
    # --- 3. Eliminate pid_pde's that have no date of rank for 'rank'.
        if THREEB and no_rank_dates:
            pre_cut_pids_num = df_wg.pid_pde.nunique()
            df_wg = cut_pids(df_wg, no_rank_dates)
            post_cut_pids_num = df_wg.pid_pde.nunique()
            print(f"   'df_wg' has {len(no_rank_dates):,} with no date-of-rank while {rank}.\n \
                They are being eliminated for a {redux_perc(pre_cut_pids_num,post_cut_pids_num)} reduction.")
            print(f"'df_wg' now has {post_cut_pids_num:,} unique pid_pde's.\n")
    # --- 4. Eliminate pid_pde's with non-RA time while 'rank'.
    if THREEC:
        print(f"---> 3. Identifying pid_pde's that don't have all RA ('R') entries for component while {rank}.")
        pids_not_r = df_wg[(df_wg.compo != 'R') & (df_wg.rank_pde == rank)]['pid_pde'].unique().tolist()
        pre_cut_pids_num = df_wg.pid_pde.nunique()
        if keep_crl:
            print(f"               note: not eliminating {crl_pid()}.")
            pids_not_r = [pid for pid in pids_not_r if pid != crl_pid()]
        df_wg = cut_pids(df_wg, pids_not_r)
        post_cut_pids_num = df_wg.pid_pde.nunique()
        print(f"   'df_wg' has {len(pids_not_r):,} pids with non-Regular Army component entries while {rank}.\n \
            They are being eliminated for a {redux_perc(pre_cut_pids_num,post_cut_pids_num)} reduction.")
        
    df_pids_num_new = df_wg.pid_pde.nunique()
    print(f""""Total reduction of the DataFrame is from {df_pids_num:,} pids to {df_pids_num_new:,} pids 
        - a reduction of {redux_perc(df_pids_num,df_pids_num_new)}.""")
    
    # --- 5. Recreate df_work from the pids left in df_wg
    df_work = df_in[df_in.pid_pde.isin(df_wg.pid_pde.unique().tolist())]
    
    # --- 6. Save 'df_work_{rank}'.
    store_feather(df_wg,f'df_work_{rank}')
    return df_work

rank = 'CPT'

if THREE:
    df_work = create_df_work(df_501_base, rank, 0)
else:
    df_work= load_feather(f'df_work_{rank}')

In [ ]:
df_work.columns

## PART THREE D: Generate work_pids_list and Table work_pids_{rank}

In [ ]:
## PART THREE D: Generate work_pids_list and Table work_pids_{rank}
def create_work_pids_list_and_table(df_work, rank, nest, show=True):
    pid_list_table_name = f'work_pids_{rank.lower()}'
    df = df_work.copy()
    df = df[['pid_pde']].drop_duplicates().sort_values(by='pid_pde').reset_index(drop=True)
    work_pids_list = df.pid_pde.tolist()
    store_json(work_pids_list,pid_list_table_name,var_dir)
    actionrdf = f"Writing pids from DataFrame 'df_work_{rank}' to Table '{pid_list_table_name}' in my schema."
    wrttab = time_start(actionrdf,nest=nest+1,show=show)
    with engine.connect() as conn:
        df.to_sql(pid_list_table_name,
                              con=conn,
                              if_exists='replace',
                              index=False,
                              method='multi')
        conn.commit()
    
    
    # Add index on snpsht_dt
    actionswtidx = "Indexing Table"
    start_idx = time_start(actionswtidx,nest=nest+2)
    sql_idx = f'CREATE INDEX idx_pid_pde_on_{pid_list_table_name} ON {pid_list_table_name}(pid_pde);'
    print_syntax(sql_idx,lang='sql')
    with engine.connect() as conn:
        conn.execute(text(sql_idx))
        conn.commit()
    
    time_stop(start_idx,action=actionswtidx, nest=nest+2)
    time_stop(wrttab,action=actionrdf, nest=nest+1)
    return work_pids_list

if THREE and THREED:
    work_pids_list = create_work_pids_list_and_table(df_work, rank, 0)

print(f"df_work has {df_work.pid_pde.nunique():,} pid_pde's")

## PART FOUR A: ID pids that have duplicate appointment dates after dropna

In [ ]:
def id_pids_dup_apnt_dt(df_in, nest, while_rank=None,store=True):
    print(f"While Rank: {while_rank}.")
    t0 = time_begin(f"Get id_pids_dup_apnt_dt for input DataFrame.", 0)
    df_w = df_in.copy()
    # now lets find pids with null ofcr_apnt_dt values while RA
    df_w = df_w[df_w.compo =='R']
    if while_rank:
        df_w = df_w[df_w.rank_pde == rank]
        print(f"df_w unique rank_pde values: {df_w.rank_pde.unique()}.")
    null_apnt_dt_pids_list = df_w[df_w['ofcr_apnt_dt'].isna()]['pid_pde'].unique().tolist()
    nul_apnt_dt_len = len(null_apnt_dt_pids_list)
    print(f"There are {nul_apnt_dt_len:,} pids with any null 'ofcr_apnt_dt' values while {while_rank + ' and ' if while_rank else ''}RA.")
    
    # now remove rows with null values for ofcr_apnt_dt
    print(f"Dropping rows (dropna) with null values for 'ofcr_apnt_dt'...")
    df_w = df_w.dropna(subset=['ofcr_apnt_dt'],axis=0)
    null_apnt_dt_pids_list = df_w[df_w['ofcr_apnt_dt'].isna()]['pid_pde'].unique().tolist()
    nul_apnt_dt_len = len(null_apnt_dt_pids_list)
    print(f"There are {nul_apnt_dt_len:,} pids with any null 'ofcr_apnt_dt' values while {while_rank + ' and ' if while_rank else ''}RA.")
    data = df_w['ofcr_apnt_dt'].tolist()
    print("df pids",Counter([type(dd) for dd in data]))

    # Now make a list of pids who still have duplicate ofcr_apnt_dt values after dropping nulls
    countr_dict2 = Counter(list(df_w[['pid_pde','ofcr_apnt_dt']].drop_duplicates().pid_pde))
    # return countr_dict2
    dup_list_apnt_dt_no_nulls = [p for p,v in countr_dict2.items() if v>1]
    print(f"There are {len(dup_list_apnt_dt_no_nulls):,} pids with duplicate dates after dropna.")
    if store:
        store_json(dup_list_apnt_dt_no_nulls,'dup_list_apnt_dt_no_nulls',var_dir,True)
    time_end(t0, 0)
    return dup_list_apnt_dt_no_nulls

## Helper Functions (mode_dict, add_promo_age_column, get_new_col)

In [ ]:
def get_mode_dict_small_pids(pid_list, rank, col, base=df_501_base):
    df = base[base.pid_pde.isin(pid_list)]
    df = df[df.rank_pde == rank]
    mode_result = df.groupby('pid_pde')[col].agg(lambda x: x.mode().iloc[0])
    print(f" after mode_result, mode_result has {len(mode_result):,} pids")
    mode_dict = mode_result.to_dict()
    print(f" after mode_dict, mode_dict has {len(mode_dict):,} pids")
    return mode_dict


def get_mode_dict_for_col(df_in, col, nest, rank=None):
    t0 = time_begin(f"Get mode_dict for '{col}'.", 0)
    df_w = df_in.copy()
    if rank:
        df_w = df_w[df_w.rank_pde == rank]
    elif col == 'ppln_pgrd_eff_dt':
        print(f"!! A Rank argument is required for a {col} mode!!")
        time_end(t0, 0)
        return
    df_w = df_w.dropna(subset=[col])
    print(f"Getting mode_dict...")
    print(f" after {col} dropna df_w has {df_w.pid_pde.nunique():,} pids")
    if col == 'ofcr_apnt_dt':
        new_col = 'yg'
        df_w[col] = df_w[col].apply(get_fy).astype(int)
        print(f" after apply(get_fy) df_w has {df_w.pid_pde.nunique():,} pids")
        print(f" after apply(get_fy) df_w has {df_w[col].nunique():,} {col}'s")
        print(f"Null test: ",[val for val in df_w[col].unique().tolist() if val !=val])
        # print(f"Set of values for df_w['{col}']:",set(df_w[col].unique()))
    print(f"Generatiog mode_result...")    
    mode_result = df_w.groupby('pid_pde')[col].agg(lambda x: x.mode().iloc[0])
    print(f" after mode_result, mode_result has {len(mode_result):,} pids")
    
    mode_dict = mode_result.to_dict()
    print(f" after mode_dict, mode_dict has {len(mode_dict):,} pids")
    
    print(f" after mode_dict, mode_dict has {df_w[col].nunique():,} unique values for '{col}'")
    print(f"Null test: ",[val for val in mode_dict.values() if val !=val])
    # print(f"Set of values for df_w['{col}']:",set(df_w[col].unique()))
    # print(f"Set of values for mode_dict:",set(mode_dict.values()))
    time_end(t0, 0)
    return mode_dict

def get_new_col(pos_dup_val_col,rank):
    if pos_dup_val_col == 'ofcr_apnt_dt':
        return 'yg'
    if pos_dup_val_col == 'ppln_pgrd_eff_dt':
        return f'dor_{rank.lower()}'

def add_promo_age_column(df_in,rank_in):
    dor_col = f'dor_{rank_in.lower()}' # This is the actual date of promotion
    age_col = f'age_{rank_in.lower()}' # This is the exact age at promotion
    print(f"Adding the '{age_col}' column to the DataFrame using the '{dor_col}' and 'date_birth_pde' columns...")
    df = df_in.copy()
    df_age = df[['pid_pde',dor_col,'date_birth_pde']].drop_duplicates().dropna(subset=[dor_col,'date_birth_pde'])
    df_age[age_col] = df_age.apply(lambda row: get_years_diff(row['date_birth_pde'], row[dor_col]), axis=1)
    age_dict = dict(zip(df_age['pid_pde'], df_age[age_col]))
    df[age_col] = df['pid_pde'].map(age_dict)
    df = move_column_after(df,age_col,dor_col)
    return df, age_col

In [ ]:
def remove_dup_pids_and_set_new_column(df_in, rank, nest, dup_list, pos_dup_val_col, save=False):
    new_col = get_new_col(pos_dup_val_col,rank)
    print(f"The new column will be '{new_col}'.")
    print(f"df_in has {df_in.pid_pde.nunique():,} unique_pids")
    print(f"df_in has {len(df_in):,} rows")    
    df = df_in.copy()
    print(f"df has {df.pid_pde.nunique():,} unique_pids")
    print(f"df has {len(df):,} rows")
    print(f"There are {len(dup_list):,} pids in the duplicate list")
    pre_cut_pids_num = df.pid_pde.nunique()
    df = cut_pids(df, dup_list)
    post_cut_pids_num = df.pid_pde.nunique()
    cut_reason = 'cutting the duplicate list'
    print(f"df after {cut_reason} has {df.pid_pde.nunique():,} unique_pids")
    print(f"df after {cut_reason}cut has {len(df):,} rows")
    print(f"Dropping rows (dropna) with null values for 'ofcr_apnt_dt'...")
    df = df.dropna(subset=['ofcr_apnt_dt'],axis=0)
    if pos_dup_val_col == 'ofcr_apnt_dt':
        print(f"df has {df.pid_pde.nunique():,} unique_pids")       
        print(f"Filtering OUT rows that are not rank_pde== {rank}.")
        df = df[df.rank_pde == rank]
        print(f"df has {df.pid_pde.nunique():,} unique_pids") 
    print(f"""   The DataFrame had {len(dup_list):,} pids eliminated for duplicate {pos_dup_val_col} values,
    for a {redux_perc(pre_cut_pids_num,post_cut_pids_num)} reduction."""
         )
    df_6 = df.copy()
    print(f"df_6 initial has {df_6.pid_pde.nunique():,} unique_pids")
    print(f"df_6 initial has {len(df_6):,} rows")
    data = df_6[pos_dup_val_col].tolist()
    # return df_6, data ########################################################################
    pid_count_dict = Counter([type(dd) for dd in data])
    print(f"Counter of {pos_dup_val_col} data types in df_6 (df_in.copy()) ",pid_count_dict)
    map_int = False
    if pos_dup_val_col == 'ofcr_apnt_dt':
        map_int = True
        print("Dropping df_6 rows with null values of pos_dup_val_col")
        df_6 = df_6.dropna(subset=[pos_dup_val_col],axis=0)
        print(f"df_6 after dropna has {df_6.pid_pde.nunique():,} unique_pids")
        print(f"df_6 after dropna has {len(df_6):,} rows")
        data = df_6[pos_dup_val_col].tolist()
        print("df_6 pids datatype count",Counter([type(dd) for dd in data]))
        print(f"Dropping duplicate rows of [[pid_pde,{pos_dup_val_col}]]")
        df_6 = df_6[['pid_pde',pos_dup_val_col]].drop_duplicates()
        print(f"df_6 (keeping only pid_pde and {pos_dup_val_col}), df_6 has {len(df_6):,} rows")
        # test for duplicate entries
        duplicate_pids = df_6.duplicated(subset=['pid_pde'],keep=False)
        duplicate_rows = df_6[duplicate_pids]
        if not duplicate_rows.empty:
            print(f"!!!! There are still duplicate values for {pos_dup_val_col}!!!!")
            return
        # Add yg column to df_6
        df_6[new_col] = df_6[pos_dup_val_col].apply(get_fy).astype(int)
        # Create yg_dict
        data = df_6[pos_dup_val_col].tolist()
        print("df pids",Counter([type(dd) for dd in data]))
        col_dict = dict(zip(df_6['pid_pde'], df_6[new_col]))
        print(f"df_6 has {df_6.pid_pde.nunique():,} unique_pids")
    
    elif pos_dup_val_col == 'ppln_pgrd_eff_dt':
        df_6 = df_6[df_6.rank_pde == rank]
        df_6 = df_6.dropna(subset=pos_dup_val_col,axis=0)
        df_6 = df_6[['pid_pde',pos_dup_val_col]].drop_duplicates()
        # test for duplicate entries
        duplicate_pids = df_6.duplicated(subset=['pid_pde'],keep=False)
        duplicate_rows = df_6[duplicate_pids]
        if not duplicate_rows.empty:
            print(f"!!!! There are still duplicate values for {pos_dup_val_col}!!!!")
            stop
        col_dict = dict(zip(df_6['pid_pde'], df_6[pos_dup_val_col]))

    else:
        print(f"!!!! Input column {pos_dup_val_col} Unknown!!!!")
        return
    
    # Add new_column to df (df_work.copy())
    data = df[pos_dup_val_col].tolist()
    print("df",Counter([type(dd) for dd in data]))
    if map_int:
        df[new_col] = df['pid_pde'].map(col_dict).astype(int)
    else:
        df[new_col] = df['pid_pde'].map(col_dict)
    df = move_column_after(df, new_col, pos_dup_val_col)
    
    return df, col_dict

## PART FOUR Execution: Assign Year Groups While CPT

In [ ]:
## HERE WE GET THE COMMISSIONING DATE for the Officers


pos_dup_val_col = 'ofcr_apnt_dt'
if FOUR:
    nest = 0
    df_gf0 = df_work.copy()
    print(f"df_gf0 (df_work.copy()) has {df_gf0.pid_pde.nunique():,} pid_pde's")
    print(f"possible duplicate values column is {pos_dup_val_col}.")
    
    if FOURB:
        ## PART FOUR A: ID pids that have duplicate appointment dates after dropna
        print(f"Deleting ID pids that have duplicate appointment dates after dropna...")
        dup_list = id_pids_dup_apnt_dt(df_gf0, 0, while_rank=rank)
        print("next")
        df_gf, df_yg_dict = remove_dup_pids_and_set_new_column(df_gf0, rank, nest, dup_list, pos_dup_val_col)
        print("--------------------> just deleted duplicate commisioning date pids")
        
    else:
        # identify the new_col for this subroutine
        df_gf = df_gf0.copy()
        new_col = get_new_col(pos_dup_val_col,rank)
        df_yg_dict = get_mode_dict_for_col(df_gf, pos_dup_val_col, nest, rank)

        print(f"original df_yg_dict has {len(df_yg_dict)} pid_pde's")
        # print("2",f"len(df_yg_dict):{len(df_yg_dict):,},",f"df_w pids: {df_w.pid_pde.nunique():,}")
        # print("yg_dict length:",len(df_yg_dict))
        print(f"Check for match df_gf and df_yg_dict")
        no_match = [pid for pid in df_gf.pid_pde.unique().tolist() if pid not in df_yg_dict]
        print(f"There are {len(no_match):,} no-match pid_pde's df_gf and  df_yg_original")
        if rank == 'CPT' and no_match:
            no_match_and_cpt = df_501_base[(df_501_base.pid_pde.isin(no_match))\
                &(df_501_base.rank_pde == 'CPT')].pid_pde.unique().tolist()
            df_check = df_501_base.dropna(subset=['ofcr_apnt_dt'])
            no_match_and_maj_checked = df_check[(df_check.pid_pde.isin(no_match))\
                &(df_check.rank_pde == 'MAJ')].pid_pde.unique().tolist()
            no_match_cpt_without_maj = [pid for pid in no_match_and_cpt if pid not in no_match_and_maj_checked]
            print(f"There are {len(no_match_and_cpt):,} pids whose only {rank} {pos_dup_val_col} row is 'NaT'")
            print(f"But {len(no_match_and_maj_checked):,} of them DO have 'MAJ' {pos_dup_val_col} rows not 'NaT'")
            print(f" So {len(no_match_and_maj_checked):,} pids are slavageable!")
            print(f"Creating df_gf5 by Removing {len(no_match_cpt_without_maj):,} unsalvageable pids from df_501_base!!!")
            print("This should be the new base df since those guys dissappeared at the last minute right after they made CPT")
            df_gf5 = cut_pids(df_501_base,no_match_cpt_without_maj)
            print(f"df_gf5 has {df_gf5.pid_pde.nunique():,} pid_pde's")
            # make a new df_gf2 cutting ALL the trouble pids to get a base yg_dict
            print(f"make a new df_gf2 cutting ALL the trouble pids to get a base yg_dict by cutting\
            all {len(no_match_and_cpt)} pids")
            # df_gf2 = cut_pids(df_gf, no_match_and_cpt)
            df_gf2 = cut_pids(df_gf.dropna(subset=[pos_dup_val_col]), no_match)
            print(f"df_gf2 has {df_gf2.pid_pde.nunique():,} pid_pde's")
            # redo the get_mode_dict
            df_yg_dict2 = get_mode_dict_for_col(df_gf2, pos_dup_val_col, nest, rank)
            print(f"df_yg_dict2 has {len(df_yg_dict2)} pid_pde's")
            no_match2 = [pid for pid in df_gf2.pid_pde.unique() if pid not in df_yg_dict2]
            print("after re-shuffle on df_gf2no_match2, no-matches:",len(no_match2))
            # make a new df_yg_dict_maj 
            print(f"make a new df_yg_dict_maj with get_mode_dict_small_pids()")
            df_yg_dict_maj = get_mode_dict_small_pids(no_match_and_maj_checked,'MAJ',pos_dup_val_col)
            print(f"df_yg_dict_maj has {len(df_yg_dict_maj)} pid_pde's")
            df_yg_dict2.update(df_yg_dict_maj)
            df_yg_dict = df_yg_dict2
            print(f"df_yg_dict has {len(df_yg_dict)} pid_pde's")
        
        no_match = [pid for pid in df_gf.pid_pde.unique().tolist() if pid not in df_yg_dict]
        print("after re-shuffle final on df_gf, no-matches:",len(no_match))
        df_gf[new_col] = df_gf['pid_pde'].map(df_yg_dict).astype(int)
        # print(f"Null test: ",[val for val in df_w[col].unique().tolist() if val !=val])
        df_gf = move_column_after(df_gf, new_col, pos_dup_val_col)
        store_feather(df_gf,'df_gf')
        print(f"The current df_gf  has {df_gf.pid_pde.nunique():,} pid_pde's")
        
#### THE OUTPUT OF THIS CELL IS THE df_gf (THE RUNNING DF FOR YG AND DOR) with 'yg' column AND THE df_yg_dict

print("-------------------------> After FOUR (Building df_yg_dict and adding the yg column):")
display(df_gf.head(5))

## PART FIVE A: ID pids that have duplicate DOR for 'rank' after dropna

In [ ]:
def id_pids_dup_pgrd_dt(df_in, rank, nest, while_compo=None,save=False):
    print(f"Rank: {rank}, nest: {nest}, while_compo: {while_compo}, save: {save}.")
    df_w = df_in.copy()
    df_w = df_w[df_w.rank_pde == rank]
    if while_compo:
        df_w = df_w[df_w.compo == while_compo]
    # Now lets find pids with null ppln_pgrd_eff_dt values for 'rank'
    null_pgrd_dt_pids_list = df_w[df_w['ppln_pgrd_eff_dt'].isna()]['pid_pde'].unique().tolist()
    nul_pgrd_dt_len = len(null_pgrd_dt_pids_list)
    print(f"There are {nul_pgrd_dt_len:,} pids with any null 'ppln_pgrd_eff_dt' values for {rank}.")
    # Now make a list of pids who still have duplicate ppln_pgrd_eff_dt values after dropping nulls
    countr_dict3 = Counter(list(df_w[['pid_pde','ppln_pgrd_eff_dt']].drop_duplicates().pid_pde))
    dup_list_pgrd_dt_no_nulls = [p for p,v in countr_dict3.items() if v>1]
    print(f"There are {len(dup_list_pgrd_dt_no_nulls):,} pids with duplicate {rank} DOR's after dropna.")
    if save:
        store_json(dup_list_pgrd_dt_no_nulls,'dup_list_pgrd_dt_no_nulls',var_dir,True)
    return dup_list_pgrd_dt_no_nulls, countr_dict3

## PART FIVE Execution: Get the Date of Rank for CPT

In [ ]:
df_gfc = df_gf.copy()
if FIVE:
    ## HERE WE GET THE DATE of RANK for the Officers as CPTs
    print(f"Getting the {rank} date of rank.")
    pos_dup_val_col = 'ppln_pgrd_eff_dt'
    new_col = get_new_col(pos_dup_val_col,rank)
    if FIVEA:
        ## PART FIVE A: ID pids that have duplicate DOR for 'rank' after dropna
        dup_list, countr_dict3 = id_pids_dup_pgrd_dt(df_gfc, rank, 0,'R')
    if FIVEB:
        print(f"Deleting ID pids that have duplicate DOR for 'rank' after dropna...")
        df_gf, dor_cpt_dict = remove_dup_pids_and_set_new_column(df_gf, rank, nest, dup_list, pos_dup_val_col)
        df_gf, age_col = add_promo_age_column(df_gf,'CPT')
        # print(df_gf.dtypes)
    else:
        dor_cpt_dict = get_mode_dict_for_col(df_gf, pos_dup_val_col, nest, rank)
        df_gf[new_col] = df_gf['pid_pde'].map(dor_cpt_dict)
        df_gf = move_column_after(df_gf, new_col, pos_dup_val_col)
        # print(df_gf.dtypes)
    store_pickle(dor_cpt_dict,'dor_cpt_dict',var_dir)
    # Start the df_pgrd_dict
    df_pgrd_dict = {pid:{'CPT': dor} for pid,dor in dor_cpt_dict.items()}
    store_pickle(df_pgrd_dict,'df_pgrd_dict', var_dir)

#### THE OUTPUT OF THIS CELL IS THE df_gf (THE RUNNING DF FOR YG AND DOR) with 'dor_cpt' column AND THE df_pgrd_dict and dor_cpt_dict

print("-------------------------> After FIVE (Building df_pgrd_dict and dor_cpt_dict and adding the dor_cpt column):")


## PART FIVE C: Find officers with Appointment dates AFTER CPT Dor

In [ ]:
if FIVEC:
    df_oadac = df_gfc.copy()
    df_oadac = df_oadac.dropna(subset = ['rank_pde','ofcr_apnt_dt','ppln_pgrd_eff_dt'])
    df_oadac = df_oadac[df_oadac.rank_pde == 'CPT']
    df_later_rows = df_oadac[df_oadac['ofcr_apnt_dt'] > df_oadac['ppln_pgrd_eff_dt']]
    bad_pids_later = df_later_rows.pid_pde.unique().tolist()
    print(f"There are {len(bad_pids_later):,} officers with dates of rank for Captain BEFORE their offcier appointment dates")
    print("Removing them from df_yg_dict and df_gf...")
    for pid in bad_pids_later:
        if pid in df_yg_dict:
            del df_yg_dict[pid]
    df_gf = cut_pids(df_gf,bad_pids_later)
    store_json(bad_pids_later,'bad_pids_later')

## PART FIVE D: Remove those pids from df_501_base, df_yg_dict, and df_gf

In [ ]:
if FIVED:
    print("Removing them from df_501_base...")
    df_501_base = cut_pids(df_501_base,bad_pids_later)

    print("Removing them from df_yg_dict...")
    for pid in bad_pids_later:
        if pid in df_yg_dict:
            del df_yg_dict[pid]
    
    print("Removing them from df_gf...")
    df_gf = cut_pids(df_gf,bad_pids_later)


## PART FIVE E: Save df_gf, work_pids_list{rank} and variables from FOUR and FIVE

In [ ]:
if FIVEE:
    store_json(df_yg_dict,'df_yg_dict',var_dir)
    store_json(df_gf.pid_pde.unique().tolist(),f'work_pids_list_{rank}',var_dir)
    store_pickle(df_pgrd_dict,'df_pgrd_dict',var_dir)
    store_feather(df_gf,'df_gf')
    
    

In [ ]:
df_gf.columns

## PART SIX A: Build df_501_working

In [ ]:
## PART SIX A: Build df_501_working
if SIXA:
    # --- 1. Establish the base set of pids from df_501_base
    df_501_base = load_feather('df_501_base')
    
    # --- 1a. If FIVEC then eliminate officers with Appointment dates AFTER CPT Dor
    if FIVEC:
        print(f"Removing the {len(bad_pids_later):,} officers with dates of rank for Captain BEFORE their offcier appointment dates")
        df_501_base = cut_pids(df_501_base, bad_pids_later)
    
    working_set = set(df_501_base.pid_pde.unique().tolist())
    print(f"The base set derived from df_501_base has {len(working_set):,} pids.")
    
    ############################  DON'T DO THIS, MAYBE COME BACK TO IT ##################################################
    # --- 2. Remove every pid not in YG 2002
    # yg = 2002
    # yg_02_set = [pid for pid,ygp in df_yg_dict.items() if ygp == yg]
    # working_set = working_set.intersection(yg_02_set)
    # print(f"Of those, we have {len(working_set)} pids that are Year Group {yg}.")
    ############################  DON'T DO THIS, MAYBE COME BACK TO IT ##################################################
    
    # --- 3. Remove every pid not in the dor_cpt_dict (the dictionary of who has a valid CPT date of rank)
    dor_cpt_dict = load_pickle('dor_cpt_dict',var_dir)
    dor_cpt_set = set(list(dor_cpt_dict.keys()))
    working_set = working_set.intersection(dor_cpt_set)
    print(f"Of those, we have {len(working_set):,} pids that have a valid date of rank for CPT.")
    
    # --- 4. Now create the df_501_working by filtering for only those pid_pde's
    print(f"Creating df_501_working by filtering df_501_base for only those officers.")   
    df_501_working = df_501_base[df_501_base.pid_pde.isin(working_set)].copy()

## PART SIX B: Map the yg, dor_cpt, age_cpt columns to df_501_working

In [ ]:
## PART SIX B: Map the yg, dor_cpt, age_cpt columns to df_501_working

if SIXB:
    # display(df_501_working.columns)
    # Map the year Group column
    print("Mapping the yg column to df_501_working...")
    df_501_working['yg'] = df_501_working['pid_pde'].map(df_yg_dict).astype(int)
    # display(df_501_working.columns)
    
    # Map the year dor_cpt column
    print("Mapping the dor_cpt column to df_501_working...")
    df_501_working['dor_cpt'] = df_501_working['pid_pde'].map(dor_cpt_dict)
    # display(df_501_working.columns)
    print("Mapping the age_cpt column to df_501_working...")
    df_501_working, age_col = add_promo_age_column(df_501_working,'CPT')
    # display(df_501_working.columns)
    print("Moving the yg column after the compo column in df_501_working...")
    df_501_working = move_column_after(df_501_working, 'yg','compo')
    # display(df_501_working.columns)
    print("Moving the dor_cpt column after the rank_pde column in df_501_working...")
    df_501_working = move_column_after(df_501_working, 'dor_cpt','rank_pde')
    # display(df_501_working.columns)
    print("Moving the age_cpt column after the dor_cpt column in df_501_working...")
    df_501_working = move_column_after(df_501_working, 'age_cpt','dor_cpt')
    # display(df_501_working.columns)
    store_feather(df_501_working,'df_501_working')

## PART SIX C: Build df_work From df_501_working with pids from df_gf

In [ ]:
## PART SIX C: Build df_work From df_501_working with pids from df_gf
if SIXC:
    df_pgrd_dict = load_pickle('df_pgrd_dict',var_dir)
    dor_cpt_dict = load_pickle('dor_cpt_dict',var_dir)
    df_work = df_501_base.copy()
    
    work_pids_list_CPT = load_json('work_pids_list_CPT',var_dir)
    df_work = df_work[df_work.pid_pde.isin(work_pids_list_CPT)]
    
    
    print("Compare work_pids_list_CPT and df_yg_dict.")
    one_not_two,two_not_one,one_and_two = list_compare(work_pids_list_CPT,list(df_yg_dict.keys()))
    print("\n\nCompare work_pids_list_CPT and dor_cpt_dict.")
    one_not_two2,two_not_one2,one_and_two2 = list_compare(work_pids_list_CPT,list(dor_cpt_dict.keys()))
    
    
    print("\n\nCompare df_yg_dict and dor_cpt_dict.")
    one_not_two3,two_not_one3,one_and_two3 = list_compare(one_and_two,one_and_two2)
    
    # df_work = df_work[df_work.pid_pde.isin(one_and_two3)]
    df_work = df_501_working[df_501_working.pid_pde.isin(one_and_two3)]
    display(df_work.columns)
    # df_work = move_column_after(df_work, 'yg','compo')
    # df_work = move_column_after(df_work, 'dor_cpt','rank_pde')


## PART SIX D: Make df_work into df_501_filtered and push into the schema as Table '501_filtered'

In [ ]:
## PART SIX D: Make df_work into df_501_filtered and push into the schema as Table '501_filtered'
nest = 0; show= True
df_501_filtered = df_work.copy()
df_501_filtered_table_name = 'table_501_filtered'
# Write the DataFrame 'df_501_filtered' to Table 'filtered_501' in my schema.
fast_copy(df_501_filtered,df_501_filtered_table_name)

# Add index on 'snpsht_dt' 'pid_pde', 'ofcr_apnt_dt', and 'ppln_pgrd_eff_dt'
actionswtidx = f"Indexing Table {df_501_filtered_table_name}"
start_idx = time_start(actionswtidx,nest=nest+2)
index_list = ['snpsht_dt','pid_pde','ofcr_apnt_dt','ppln_pgrd_eff_dt','age_exact','asg_uic_pde']
sql_idx = ''
for col in index_list:
    sql_idx += "\r" + f"""CREATE INDEX idx_{col}_on_{df_501_filtered_table_name} ON {df_501_filtered_table_name}({col});"""
print_syntax(sql_idx,lang='sql')
with engine.connect() as conn:
    conn.execute(text(sql_idx))
    conn.commit()

time_stop(start_idx,action=actionswtidx, nest=nest+2)

check_table(df_501_filtered_table_name)

store_feather(df_501_filtered,'df_501_filtered')

## PART SEVEN A: Visualize CPTs Who Became MAJ by YG (Count)

In [ ]:
## PART SEVEN A: Visualize CPTs Who Became MAJ by YG

df_8 = df_501_filtered.copy()
# Get CPTs by yg
df_cpt_summary = df_8[['pid_pde','yg']].drop_duplicates()

# Identify those who ever became MAJ
pids_with_maj = set(df_8.loc[df_8['rank_pde'] == 'MAJ', 'pid_pde'])

# Mark them
df_cpt_summary['made_maj'] = df_cpt_summary['pid_pde'].isin(pids_with_maj)

# group by yg
summary_base = df_cpt_summary.groupby('yg')['made_maj'].agg(['sum','count'])
summary= summary_base.loc[2002:2016].copy()
summary['percent'] = (summary['sum'] / summary['count'] * 100).round(2)

# convert summary.index to list of years
# years = summary.index.tolist()
years = summary.index.tolist()

# Determine which years to label (every nth year)
n = 2
labels = [str(int(year)) if year % n == 0 else '' for year in years]

# Plot Count
plt.figure(figsize=(10,4))
plt.bar(years, summary['sum'], color = 'steelblue')
plt.title(f'CPTs Who Became MAJ by YG {int(min(years))}-{int(max(years))}')
plt.ylabel('Count')
plt.xlabel('Year Group (YG)')
plt.xticks(ticks=years, labels=labels, rotation=45)
plt.tick_params(axis='x', which='major', length=10)
plt.tick_params(axis='x', which='minor', length=4)
plt.tight_layout()
plt.show()

## PART SEVEN B: Visualize CPTs Who Became MAJ by YG (Percent)

In [ ]:
## PART SEVEN B: Visualize CPTs Who Became MAJ by YG (Percent)
# Plot Percent
plt.figure(figsize=(10,4))
plt.bar(years, summary['percent'], color = 'seagreen')
plt.title('% of CPTs Who Became MAJ by YG')
plt.ylabel('Percent')
plt.xlabel('Year Group (YG)')
plt.xticks(ticks=years, labels=labels, rotation=45)
plt.tick_params(axis='x', which='major', length=10)
plt.tick_params(axis='x', which='minor', length=4)
plt.tight_layout()
plt.show()

make the machine by showing the correlation of a pid_pde made captain with age, or with married, or MOS
you can just say if the pid is still in the army at the primary zone consideration they failed to make it, and if they left before then they can be censored.  Or maybe 2 years before.


## PART EIGHT: Add Major Promotion Date and Major Age Columns

In [ ]:
## PART EIGHT: Add Major Promotion Date and Major Age Columns
def extract_maj_dor(df, nest):
    t0=time_begin("extract_maj_dor", nest)
    df_sorted = df.sort_values(by=["pid_pde", "snpsht_dt"])
    maj_dates = (
        df_sorted[df_sorted["rank_pde"] == "MAJ"]
        .groupby("pid_pde")
        .first()["ppln_pgrd_eff_dt"]
    )
    df_result = df.copy()
    df_result["dor_maj"] = df_result["pid_pde"].map(maj_dates)
    df_result = move_column_after(df_result,'dor_maj','age_cpt')
    time_end(t0,nest)
    return df_result
    
if EIGHT:
    print("== PART EIGHT: Add Major Promotion Date Column ==")
    df_501_major = extract_maj_dor(df_work, 0)
    print("After dor_maj column:",df_501_major.columns)
    df_501_major, age_col = add_promo_age_column(df_501_major,'MAJ')
    print(f"After {age_col} column:",df_501_major.columns)
    store_feather(df_501_major, 'df_501_major')
else:
    df_501_major = load_feather('df_501_major')

In [ ]:
### Test Cell
b_dict = {'a':1,'b':2,'c':3,'d':4}
display(b_dict)
def get_diffy(one,two):
    return one-two
dg = pd.DataFrame({'A':['a','b','c','d'],'B':[1,1,1,1]})
# display(dg)
# test the get_diffy function
get_diffy(b_dict['a'],9)
# now use it to create a new column in dg
dg['C'] = dg.apply(lambda row: get_diffy(b_dict[row['A']], row['B']), axis=1)
dg

## OLD CODE

In [ ]:
# ### THIS IS THE SCRATCHWORK 
# ### FOR THE ABOVE FUNCTION remove_dup_pids_and_set_new_column

# dup_list=dup_list_pgrd_dt_no_nulls
# pos_dup_val_col='ppln_pgrd_eff_dt'

# nest = 0
# df = df_work.copy()
# pre_cut_pids_num = df.pid_pde.nunique()
# df = cut_pids(df, dup_list)
# post_cut_pids_num = df.pid_pde.nunique()
# print(f"""   The DataFrame has {len(dup_list):,} pids being eliminated for duplicate {pos_dup_val_col} values,
#                     for a {redux_perc(pre_cut_pids_num,post_cut_pids_num)} reduction."""
#          )
# df_99 = df.copy()

# if pos_dup_val_col == 'ppln_pgrd_eff_dt':
#     new_col = 'dor_'+rank.lower()
#     df_99 = df_99[df_99.rank_pde == rank]
#     df_99 = df_99.dropna(subset=pos_dup_val_col,axis=0)
#     df_99 = df_99[['pid_pde',pos_dup_val_col]].drop_duplicates()
#     # test for duplicate entries
#     duplicate_pids = df_99.duplicated(subset=['pid_pde'],keep=False)
#     duplicate_rows = df_99[duplicate_pids]
#     if not duplicate_rows.empty:
#         print(f"!!!! There are still duplicate values for {pos_dup_val_col}!!!!")
#         stop
#     col_dict = dict(zip(df_99['pid_pde'], df_99[pos_dup_val_col]))

    
# # Add new_column to df (df_work.copy())
# df[new_col] = df['pid_pde'].map(col_dict)
# df = move_column_after(df, new_col, pos_dup_val_col)
# # return df, col_dict


In [ ]:
### Remove pids that have duplicate appointment dates even after dropping nulls


# OLD MAYBE?????????????????????????????
# set_apnt_list = set(dup_list_apnt_dt_no_nulls)
# set_pgrd_list = set(dup_list_pgrd_dt_no_nulls)
# count = len(set_apnt_list.intersection(set_pgrd_list))
# remove_pids_set = set_apnt_list | set_pgrd_list
# print(len(set_apnt_list)+len(set_pgrd_list))
# print(count)
# print(len(remove_pids_set))

# # make df_5 by removing duplicate apnt date pids and pgrd_dt pids
# df_len = len(df_work)
# print("---> Creating df_5 by eliminating pids with ambiguous dates of rank and appointment dates.")
# df_len = len(df_work)
# print(f"'df_work' has {df_len:,} rows.")
# len_pids_df_work = len(df_work.pid_pde.unique().tolist())
# print(f"df_work currently has {len_pids_df_work:,} unique pid_pde's.") 
# df_5 =  df_work.copy()
# print(f"Cutting (cut_pids()) {len(remove_pids_set):,} pids with ambiguous dates of rank and appointment dates")
# df_5 = cut_pids(df_work.copy(),remove_pids_set)
# print(f"df_5 Now has {len(df_5):,} rows after dropping {df_len-len(df_5):,} rows.")
# pids_df_5 = len_pids_df_5 = len(df_5.pid_pde.unique().tolist())
# pids_diff = len_pids_df_work - len_pids_df_5
# print(f"df_5 Now has {len_pids_df_5:,} unique pid_pde's after dropping {pids_diff :,} pids.") 
# print(f"For a reduction of {(pids_diff)*100/len_pids_df_work:,.2f} %.") 

# # make df_6 by removing all null dates of rank and appointment dates
# df_6 = df_5.copy()
# df_6 = df_6.dropna(subset='ppln_pgrd_eff_dt',axis=0)
# df_6 = df_6.dropna(subset='ofcr_apnt_dt',axis=0)

# # Add yg column to df_6
# df_6['yg'] = df_6['ofcr_apnt_dt'].apply(get_fy)
# df_6 = move_column_after(df_6,'yg','ofcr_apnt_dt')

# # Create yg_dict and dor_dicts for year group and date of rank
# df_yg_dict = dict(zip(df_6['pid_pde'], df_6['yg']))
# df_dor_cpt_dict = dict(zip(df_6['pid_pde'], df_6['ppln_pgrd_eff_dt']))

# df_501_filtered_pids_list = list(df_5.pid_pde.unique())
# df_5_pids_list = list(df_5.pid_pde.unique())
# print(f"df_501_fdf_501_filtered_pids_list has {len(df_501_filtered_pids_list):,} pids.")
# print(f"df_5_pids_list has {len(df_5_pids_list):,} pids.")
# diff_set = set(df_5_pids_list).difference(set(list(df_501_filtered_pids_list)))
# count = len(diff_set)
# print(f"For a differenece of {count:,} unique pids")
# new_list = list(set(df_501_filtered_pids_list) - set(df_5_pids_list))
# print(f"However the list of items in df_501_filtered_pids_list not in the dictionaries is {new_list}.")

# # Now let's make the final df_501_filtered dataframe
# df_7 = df_5[['pid_pde']].drop_duplicates()

# df_8 = df_501_base[df_501_base['pid_pde'].isin(df_7['pid_pde'])].copy()
# display(df_8.columns)

# #  Now add a 'yg' column and put it right after 'ofcr_apnt_dt'
# df_8['yg'] = df_8['pid_pde'].map(df_yg_dict)
# df_8 = move_column_after(df_8, 'yg', 'ofcr_apnt_dt')
# display(df_8.columns)

# #  Now add a 'dor_cpt' column and put it right after 'ppln_pgrd_eff_dt'
# df_8['dor_cpt'] = df_8['pid_pde'].map(df_dor_cpt_dict)
# df_8 = move_column_after(df_8, 'dor_cpt', 'ppln_pgrd_eff_dt')
# display(df_8.columns)

# print(f"df_8 has {len(df_8.pid_pde.unique()):,} pids.")

In [ ]:
# for pp in dup_list_apnt_dt_no_nulls[:10]:
#     display(df_501_base[(df_501_base.pid_pde == pp)&(df_501_base.rank_pde == 'CPT')]\
#     .sort_values(by=['pid_pde','snpsht_dt'],ascending=True)\
#     [['pid_pde','snpsht_dt','rank_pde','ppln_pgrd_eff_dt','ofcr_apnt_dt','compo','occ_crer_grp_cd','pri_dod_occ_cd','pn_age_qy']].\
#     # drop_duplicates(subset=['pid_pde','ofcr_apnt_dt'],keep='first').\
#             head(30))

In [ ]:
# for pp in dup_list_pgrd_dt_no_nulls[:5]:
#     display(df_501_base[(df_501_base.pid_pde == pp)&(df_501_base.rank_pde == 'CPT')]\
#     .sort_values(by=['pid_pde','snpsht_dt'],ascending=True)\
#     [['pid_pde','snpsht_dt','rank_pde','ppln_pgrd_eff_dt','ofcr_apnt_dt','compo','occ_crer_grp_cd','pri_dod_occ_cd','pn_age_qy']].\
#     # drop_duplicates(subset=['pid_pde','ofcr_apnt_dt'],keep='first').\
#             head(30))

In [ ]:
##EXPERIMENTS
# Now let's get a list of all pid_pde's that don't have an entry for 'OOO' by getting
# The last snapshot row for each pid_pde
# df_work = df_8.copy()
# idx = df_work.groupby('pid_pde')['snpsht_dt'].idxmax()

# df_last_snap = df_work.loc[idx].reset_index(drop=True)
# df_last_snap.rank_pde.unique()

# pids_2lt = set(df_8[df_8['rank_pde'] == '2LT']['pid_pde'])
# pids_1lt = set(df_8[df_8['rank_pde'] == '1LT']['pid_pde'])
# pids_cpt = set(df_8[df_8['rank_pde'] == 'CPT']['pid_pde'])
# pids_maj = set(df_8[df_8['rank_pde'] == 'MAJ']['pid_pde'])
# pids_ooo = set(df_8[df_8['rank_pde'] == 'OOO']['pid_pde'])

# pids_all = list(pids_2lt & pids_1lt & pids_cpt & pids_maj & pids_ooo)
# pids_1235 = list(pids_2lt & pids_1lt & pids_cpt & pids_ooo)
# print(len(pids_all))
# print(len(pids_1235))
# print(set(pids_1235).difference(set(pids_all)))

In [ ]:
CA = ['IN','AR','EN','AV','FA','SF','AD'] # 7 branches
CS = ['CM','CA','MI','MP','SC'] # 5 branches
CSS = ['AG','FI','OD','QM','TC','LG'] # 6 branches
dd = ['IN']
branch_option = 'CBTA'
if branch_option:
    if branch_option == 'CBTA':
        branch_txt = branch_option
        branch_option = CA  
print(branch_txt)
print(branch_option)